# ChatUniTest LoRA Fine-tuning

**Goal**: Fine-tune Qwen2.5-Coder-7B-Instruct with QLoRA to generate Java JUnit tests

**Before running**:
- Make sure your HuggingFace token (write access) is ready

## Step 1: Install Dependencies

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

%pip install -q transformers==4.44.0 peft==0.11.1 trl==0.9.6 \
    "bitsandbytes>=0.44.0" accelerate==0.33.0 \
    datasets sentencepiece huggingface_hub

# Verify GPU
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Step 2: Login to HuggingFace (write token required)

For our training to work, you will need to connect to HuggingFace with a token that has write access.

In [ ]:
from huggingface_hub import login, notebook_login

# Option 1
notebook_login()

# Option 2
# login(token="hf_xxxxxxxxxxxxx")

## Step 3: Prepare Dataset

In [ ]:
from datasets import load_dataset, Dataset
import pandas as pd
import re

PROMPT_TEMPLATE = """mode=COMPLETION
projectPath=unknown
assertionStyle=JUNIT
staticSnapshot:
{context}
runtimeFacts:

### JUnit Test:
"""

def build_full_text(context: str, test: str) -> str:
    return PROMPT_TEMPLATE.format(context=context.strip()) + test.strip()

def has_assertion(test: str) -> bool:
    return any(kw in test for kw in ["assert", "Assert", "verify", "Verify", "fail("])

def is_valid_java(code: str) -> bool:
    return code.count("{") > 0 and abs(code.count("{") - code.count("}")) <= 2

def estimate_tokens(text: str) -> int:
    return len(text) // 4

# Load dataset
print("Loading dataset...")
raw = load_dataset("zzzghttt/context2test", split="train")
print(f"Raw samples: {len(raw)}")
print(f"Columns: {raw.column_names}")

# Auto-detect column names
col_context = next((c for c in ["context", "input", "source"] if c in raw.column_names), None)
col_test    = next((t for t in ["test", "output", "target"] if t in raw.column_names), None)
print(f"Using: context='{col_context}', test='{col_test}'")

df = raw.to_pandas()[[col_context, col_test]].copy()
df.columns = ["context", "test"]

# ── Data cleaning ──
before = len(df)
df = df.drop_duplicates(subset=["context"])
print(f"After dedup: {len(df)} (removed {before - len(df)})")

df = df[df["test"].apply(has_assertion)]
print(f"After assertion filter: {len(df)}")

df = df[df["test"].apply(is_valid_java)]
print(f"After Java structure filter: {len(df)}")

df = df[df["context"].str.strip().str.len() > 30]
print(f"After empty context filter: {len(df)}")

# Build full training text
df["text"] = df.apply(lambda r: build_full_text(r["context"], r["test"]), axis=1)
df["token_est"] = df["text"].apply(estimate_tokens)
df = df[df["token_est"] <= 2048]
print(f"After token length filter: {len(df)}")

# Sample top 5000
df = df.sample(frac=1, random_state=42).reset_index(drop=True).head(5000)
print(f"\nFinal training samples: {len(df)}")

# Convert to HuggingFace Dataset
train_dataset = Dataset.from_pandas(df[["text"]])

# Preview first sample
print("\n=== Sample preview (first 500 chars) ===")
print(train_dataset[0]["text"][:500])

## Step 4: Load Base Model (QLoRA 4-bit)

In [ ]:
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)

BASE_MODEL = "Qwen/Qwen2.5-Coder-7B-Instruct"

# 4-bit QLoRA configuration
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Loading model (4-bit)...")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
model.config.use_cache = False
model.config.pretraining_tp = 1

print(f"Model loaded. VRAM usage: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## Step 5: Configure LoRA

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Prepare model for QLoRA training
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=64,                    
    lora_alpha=64,           
    target_modules=[
        "q_proj", "v_proj",
        "k_proj", "o_proj",
    ],
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## Step 6: Train
Replace HF_USERNAME with your HuggingFace username before running this step.

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

# ── Set your HuggingFace username here ──
HF_USERNAME = "your-hf-username"   # <-- REPLACE THIS
OUTPUT_MODEL = f"{HF_USERNAME}/my-testgen-lora"

training_args = TrainingArguments(
    output_dir="./my-testgen-lora",
    num_train_epochs=2,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=8,       # effective batch_size = 32
    gradient_checkpointing=True,        
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    fp16=True,                           
    bf16=False,
    logging_steps=10,
    save_strategy="steps",
    save_steps=100,                     
    save_total_limit=3,
    report_to="none",                   
    optim="paged_adamw_32bit",          
    max_grad_norm=0.3,
    group_by_length=True,                # group similar-length samples to reduce padding
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    args=training_args,
    tokenizer=tokenizer,
    dataset_text_field="text",
    max_seq_length=2048,                 # must match inference cutoff_len
    packing=False,
)

print("Starting training...")
print(f"Total steps: {len(train_dataset) // (training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps)}")
trainer.train()

## Step 7: Save and Push to HuggingFace Hub

In [ ]:
print("Saving model...")
trainer.save_model("./my-testgen-lora")

print(f"Pushing to HuggingFace Hub: {OUTPUT_MODEL}")
model.push_to_hub(OUTPUT_MODEL, private=False)
tokenizer.push_to_hub(OUTPUT_MODEL, private=False)

print(f"\nDone! Model available at: https://huggingface.co/{OUTPUT_MODEL}")
print(f"\nNext step — update model_server.py line 23:")
print(f'  PeftModel.from_pretrained(model, "{OUTPUT_MODEL}")')

## Step 8 (Optional): Resume from Checkpoint

If Colab disconnects, use this cell to resume from the latest checkpoint.

In [ ]:
# import os

# # Find latest checkpoint
# checkpoints = [
#     d for d in os.listdir("./my-testgen-lora")
#     if d.startswith("checkpoint-")
# ]
# if checkpoints:
#     latest = sorted(checkpoints, key=lambda x: int(x.split("-")[1]))[-1]
#     resume_from = f"./my-testgen-lora/{latest}"
#     print(f"Resuming from {resume_from}")
#     trainer.train(resume_from_checkpoint=resume_from)
# else:
#     print("No checkpoint found — please run from Step 1")